# Phase 19 — ATS Friendliness Benchmark and Scorer

This notebook creates a locked ATS benchmark manifest, parser benchmark metrics, issue-label taxonomy, transparent ATS scorer, release-gate metrics, issue-level errors, and safe fallback behavior for `atsFriendliness`.

The benchmark uses deterministic synthetic CV fixtures because no raw CV files are stored in this repository. The fixtures contain only non-personal text and parser metadata needed to test behavior. Real CV documents can replace these fixtures later by preserving the same manifest schema.


## Step 19.1 — Locked CV benchmark manifest

### Purpose
Create a locked benchmark manifest covering normal PDF, scanned PDF, multi-column PDF, table-heavy PDF, short CV, long CV, and supported DOCX cases.

### Required input
Phase 7 ATS scoring policy, Model API CV analyzer boundary, and synthetic CV fixture definitions with parser metadata.

### Action
Build deterministic benchmark rows with case family, file family, support status, extracted text, parser signals, and reviewer labels. Save the manifest and fixture table under `artifacts/ats_benchmark/` and `reports/`.

### Expected output
A locked manifest with SHA-256 hashes, required case coverage, label version, and source fixture inventory.

### Verification
Every required case family exists, each fixture has labels, unsupported/scanned cases are explicitly represented, and no raw personal data is stored.


In [1]:
from __future__ import annotations

import hashlib
import json
import math
import re
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

PHASE_ID = "phase_19_ats_friendliness_benchmark_scorer"
SCHEMA_VERSION = "ats-benchmark-scorer-v1"
LABEL_VERSION = "ats-issue-labels-v1"
SEED = 202619
RELEASE_THRESHOLDS = {
    "macro_issue_precision": 0.85,
    "macro_issue_recall": 0.85,
    "bucket_agreement": 0.80,
    "empty_text_rate_supported_max": 0.20,
    "critical_parse_failure_recall": 1.0,
}

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "TODOS.md").exists():
    for parent in Path.cwd().resolve().parents:
        if (parent / "TODOS.md").exists():
            REPO_ROOT = parent
            break

REPORTS_DIR = REPO_ROOT / "reports"
ARTIFACTS_DIR = REPO_ROOT / "artifacts"
BENCHMARK_DIR = ARTIFACTS_DIR / "ats_benchmark"
BENCHMARK_CSV_PATH = BENCHMARK_DIR / "phase_19_cv_benchmark.csv"
BENCHMARK_MANIFEST_PATH = REPORTS_DIR / "phase_19_cv_benchmark_manifest.json"
PARSER_METRICS_PATH = REPORTS_DIR / "phase_19_parser_benchmark_metrics.json"
ISSUE_LABELS_PATH = REPORTS_DIR / "phase_19_ats_issue_labels.json"
SCORER_METRICS_PATH = REPORTS_DIR / "phase_19_ats_scorer_metrics.json"
ERRORS_FALLBACKS_PATH = REPORTS_DIR / "phase_19_issue_errors_fallbacks.json"
PHASE_REPORT_PATH = REPORTS_DIR / "phase_19_ats_friendliness_benchmark_scorer.json"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
BENCHMARK_DIR.mkdir(parents=True, exist_ok=True)
np.random.seed(SEED)


def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


def rel(path: Path) -> str:
    return str(path.resolve().relative_to(REPO_ROOT))


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(payload, indent=2, sort_keys=True, default=str) + "\n", encoding="utf-8")


def text_for(case_family: str, variant: int) -> str:
    base = {
        "normal_pdf": "Summary Backend developer. Experience Built REST APIs and improved latency by 35%. Skills Python SQL Docker. Education Computer Science. Projects API platform.",
        "multi_column_pdf": "Summary Data analyst Skills Python SQL Tableau Experience Built dashboards Revenue grew 12% Education Statistics",
        "table_heavy_pdf": "Summary Software engineer Skills Java SQL Experience table row cell project cell achievement Education Bachelor",
        "short_cv": "Summary Junior developer. Skills HTML CSS. Education Bootcamp.",
        "long_cv": "Summary Senior engineer. Experience " + "Managed projects and improved uptime by 45%. " * 80 + "Skills Java Python SQL. Education Engineering.",
        "docx": "Summary Product analyst. Experience Improved reporting accuracy by 22%. Skills SQL Excel Python. Education Business.",
        "scanned_pdf": "",
    }
    return base[case_family] + (f" Fixture {variant}." if base[case_family] else "")

required_case_families = ["normal_pdf", "scanned_pdf", "multi_column_pdf", "table_heavy_pdf", "short_cv", "long_cv", "docx"]
case_config = {
    "normal_pdf": {"file_family": "pdf", "supported": True, "layout_risk": "low", "page_count": 1, "expected": []},
    "scanned_pdf": {"file_family": "pdf", "supported": True, "layout_risk": "critical", "page_count": 2, "expected": ["empty_parse_risk", "parseability_issue"]},
    "multi_column_pdf": {"file_family": "pdf", "supported": True, "layout_risk": "high", "page_count": 2, "expected": ["formatting_risk_issue"]},
    "table_heavy_pdf": {"file_family": "pdf", "supported": True, "layout_risk": "high", "page_count": 2, "expected": ["formatting_risk_issue", "metric_evidence_issue"]},
    "short_cv": {"file_family": "pdf", "supported": True, "layout_risk": "medium", "page_count": 1, "expected": ["section_completeness_issue", "metric_evidence_issue"]},
    "long_cv": {"file_family": "pdf", "supported": True, "layout_risk": "medium", "page_count": 6, "expected": ["formatting_risk_issue"]},
    "docx": {"file_family": "docx", "supported": True, "layout_risk": "medium", "page_count": 1, "expected": []},
}

rows = []
for case_family in required_case_families:
    for variant in range(1, 4):
        cfg = case_config[case_family]
        extracted_text = text_for(case_family, variant)
        word_count = len(re.findall(r"\w+", extracted_text))
        section_hits = sum(1 for key in ["summary", "experience", "skills", "education", "projects"] if key in extracted_text.lower())
        table_density = 0.55 if case_family == "table_heavy_pdf" else 0.05
        column_count = 2 if case_family == "multi_column_pdf" else 1
        parser_error = case_family == "scanned_pdf"
        unsupported = False
        labeled_issues = list(cfg["expected"])
        if case_family == "scanned_pdf":
            labeled_issues = ["parseability_issue", "empty_parse_risk"]
        label_score = max(0, 100 - 25 * len(labeled_issues) - (20 if parser_error else 0))
        label_bucket = "high" if label_score >= 75 else "medium" if label_score >= 45 else "low"
        rows.append({
            "fixture_id": f"ATS19-{case_family}-{variant:02d}",
            "case_family": case_family,
            "file_family": cfg["file_family"],
            "supported_file_type": cfg["supported"],
            "layout_risk": cfg["layout_risk"],
            "page_count": cfg["page_count"],
            "extracted_text": extracted_text,
            "expected_character_count": max(len(extracted_text), 250 if case_family == "scanned_pdf" else len(extracted_text)),
            "extracted_character_count": len(extracted_text),
            "word_count": word_count,
            "section_count": section_hits,
            "column_count": column_count,
            "table_density": table_density,
            "parser_error": parser_error,
            "ocr_required": case_family == "scanned_pdf",
            "unsupported_file_type": unsupported,
            "label_issue_keys": json.dumps(sorted(labeled_issues)),
            "label_ats_score": label_score,
            "label_ats_bucket": label_bucket,
            "label_version": LABEL_VERSION,
        })

benchmark_df = pd.DataFrame(rows).sort_values(["case_family", "fixture_id"]).reset_index(drop=True)
benchmark_df.to_csv(BENCHMARK_CSV_PATH, index=False)
case_counts = benchmark_df["case_family"].value_counts().sort_index().to_dict()
missing_cases = sorted(set(required_case_families) - set(case_counts))
manifest = {
    "phase_id": PHASE_ID,
    "schema_version": SCHEMA_VERSION,
    "label_version": LABEL_VERSION,
    "generated_at": utc_now(),
    "status": "complete" if not missing_cases else "blocked",
    "artifact": {"path": rel(BENCHMARK_CSV_PATH), "sha256": sha256_file(BENCHMARK_CSV_PATH), "row_count": int(len(benchmark_df))},
    "required_case_families": required_case_families,
    "case_counts": {str(k): int(v) for k, v in case_counts.items()},
    "missing_cases": missing_cases,
    "privacy": {"contains_raw_personal_cv_data": False, "fixture_type": "synthetic_non_personal"},
    "locked": True,
}
write_json(BENCHMARK_MANIFEST_PATH, manifest)
manifest


{'phase_id': 'phase_19_ats_friendliness_benchmark_scorer',
 'schema_version': 'ats-benchmark-scorer-v1',
 'label_version': 'ats-issue-labels-v1',
 'generated_at': '2026-06-02T04:52:49.317125+00:00',
 'status': 'complete',
 'artifact': {'path': 'artifacts/ats_benchmark/phase_19_cv_benchmark.csv',
  'sha256': 'e28618309571fd5e9c58b557601520e54efd59e5b028d481a3fb031696b31f17',
  'row_count': 21},
 'required_case_families': ['normal_pdf',
  'scanned_pdf',
  'multi_column_pdf',
  'table_heavy_pdf',
  'short_cv',
  'long_cv',
  'docx'],
 'case_counts': {'docx': 3,
  'long_cv': 3,
  'multi_column_pdf': 3,
  'normal_pdf': 3,
  'scanned_pdf': 3,
  'short_cv': 3,
  'table_heavy_pdf': 3},
 'missing_cases': [],
 'privacy': {'contains_raw_personal_cv_data': False,
  'fixture_type': 'synthetic_non_personal'},
 'locked': True}

## Step 19.2 — Parser benchmark metrics

### Purpose
Measure extraction coverage, empty-text rate, section detection accuracy, and critical parse failure recall.

### Required input
Locked benchmark fixture table with parser metadata and reviewer labels.

### Action
Compute parser-level metrics overall and by case family. Critical parse failures are scanned, empty parse, unsupported, or parser-error cases.

### Expected output
`reports/phase_19_parser_benchmark_metrics.json` with coverage, empty-text, section detection, and failure-recall metrics.

### Verification
Supported-document empty-text rate is measured separately; scanned and empty cases are not hidden from aggregate reporting.


In [2]:
def parse_issues(row: pd.Series) -> list[str]:
    issues: list[str] = []
    coverage = row["extracted_character_count"] / max(1, row["expected_character_count"])
    text = str(row["extracted_text"]).lower()
    if row["unsupported_file_type"]:
        issues.append("unsupported_file_type")
    if row["parser_error"] or row["extracted_character_count"] == 0 or coverage < 0.10:
        return ["empty_parse_risk", "parseability_issue"]
    if row["section_count"] <= 3:
        issues.append("section_completeness_issue")
    if not re.search(r"\b\d+%|\b\d+x|\b\d+\+", text):
        issues.append("metric_evidence_issue")
    if row["column_count"] > 1 or row["table_density"] >= 0.40 or row["word_count"] > 450:
        issues.append("formatting_risk_issue")
    return sorted(set(issues))

benchmark_df["predicted_parser_issues"] = benchmark_df.apply(parse_issues, axis=1)
benchmark_df["extraction_coverage"] = benchmark_df["extracted_character_count"] / benchmark_df["expected_character_count"].clip(lower=1)
benchmark_df["empty_text"] = benchmark_df["extracted_character_count"] == 0
benchmark_df["section_detected"] = benchmark_df["section_count"] >= 3
benchmark_df["label_section_complete"] = ~benchmark_df["label_issue_keys"].map(lambda s: "section_completeness_issue" in json.loads(s))
benchmark_df["critical_parse_failure_label"] = benchmark_df["label_issue_keys"].map(lambda s: bool({"parseability_issue", "empty_parse_risk", "unsupported_file_type"} & set(json.loads(s))))
benchmark_df["critical_parse_failure_pred"] = benchmark_df["predicted_parser_issues"].map(lambda xs: bool({"parseability_issue", "empty_parse_risk", "unsupported_file_type"} & set(xs)))

supported = benchmark_df[benchmark_df["supported_file_type"]]
critical_labels = benchmark_df["critical_parse_failure_label"].sum()
critical_tp = ((benchmark_df["critical_parse_failure_label"]) & (benchmark_df["critical_parse_failure_pred"])).sum()
parser_metrics = {
    "phase_id": PHASE_ID,
    "schema_version": SCHEMA_VERSION,
    "generated_at": utc_now(),
    "overall": {
        "row_count": int(len(benchmark_df)),
        "mean_extraction_coverage": float(benchmark_df["extraction_coverage"].mean()),
        "empty_text_rate_all": float(benchmark_df["empty_text"].mean()),
        "empty_text_rate_supported": float(supported["empty_text"].mean()) if len(supported) else None,
        "section_detection_accuracy": float((benchmark_df["section_detected"] == benchmark_df["label_section_complete"]).mean()),
        "critical_parse_failure_recall": float(critical_tp / critical_labels) if critical_labels else None,
        "critical_parse_failure_label_count": int(critical_labels),
    },
    "by_case_family": {},
}
for case_family, group in benchmark_df.groupby("case_family", sort=True):
    parser_metrics["by_case_family"][case_family] = {
        "row_count": int(len(group)),
        "mean_extraction_coverage": float(group["extraction_coverage"].mean()),
        "empty_text_rate": float(group["empty_text"].mean()),
        "section_detection_accuracy": float((group["section_detected"] == group["label_section_complete"]).mean()),
    }
write_json(PARSER_METRICS_PATH, parser_metrics)
parser_metrics


{'phase_id': 'phase_19_ats_friendliness_benchmark_scorer',
 'schema_version': 'ats-benchmark-scorer-v1',
 'generated_at': '2026-06-02T04:52:49.348915+00:00',
 'overall': {'row_count': 21,
  'mean_extraction_coverage': 0.8571428571428571,
  'empty_text_rate_all': 0.14285714285714285,
  'empty_text_rate_supported': 0.14285714285714285,
  'section_detection_accuracy': 0.7142857142857143,
  'critical_parse_failure_recall': 1.0,
  'critical_parse_failure_label_count': 3},
 'by_case_family': {'docx': {'row_count': 3,
   'mean_extraction_coverage': 1.0,
   'empty_text_rate': 0.0,
   'section_detection_accuracy': 1.0},
  'long_cv': {'row_count': 3,
   'mean_extraction_coverage': 1.0,
   'empty_text_rate': 0.0,
   'section_detection_accuracy': 1.0},
  'multi_column_pdf': {'row_count': 3,
   'mean_extraction_coverage': 1.0,
   'empty_text_rate': 0.0,
   'section_detection_accuracy': 1.0},
  'normal_pdf': {'row_count': 3,
   'mean_extraction_coverage': 1.0,
   'empty_text_rate': 0.0,
   'section_

## Step 19.3 — ATS issue taxonomy labels

### Purpose
Lock the issue taxonomy for parseability, section completeness, contact detection, date detection, metric evidence, formatting risk, and empty parse risk.

### Required input
Benchmark labels and Phase 7 issue taxonomy policy.

### Action
Define durable issue keys, label descriptions, positive counts, and evidence requirements.

### Expected output
`reports/phase_19_ats_issue_labels.json` with taxonomy and benchmark label distribution.

### Verification
Every required taxonomy key exists and each predicted issue must map to an evidence rule.


In [3]:
issue_taxonomy = {
    "parseability_issue": {"description": "Parser cannot reliably extract meaningful text.", "evidence": ["parser_error", "low extraction coverage", "empty text"]},
    "section_completeness_issue": {"description": "Core CV sections are missing or not detected.", "evidence": ["section_count < 3", "missing summary/experience/skills/education"]},
    "contact_detection_issue": {"description": "Contact fields are absent or unverified.", "evidence": ["email/phone signal missing when raw text is available"]},
    "date_detection_issue": {"description": "Experience or education dates are absent or inconsistent.", "evidence": ["date pattern missing", "timeline ambiguity"]},
    "metric_evidence_issue": {"description": "Experience lacks quantified impact evidence.", "evidence": ["no percent/count/multiplier evidence"]},
    "formatting_risk_issue": {"description": "Layout may harm ATS parsing.", "evidence": ["multi-column", "table-heavy", "excessive length"]},
    "empty_parse_risk": {"description": "Extracted text is empty or near-empty.", "evidence": ["zero extracted characters", "OCR required"]},
    "unsupported_file_type": {"description": "File type is not accepted by current parser contract.", "evidence": ["unsupported extension or mime type"]},
}
required_issue_keys = [
    "parseability_issue", "section_completeness_issue", "contact_detection_issue", "date_detection_issue",
    "metric_evidence_issue", "formatting_risk_issue", "empty_parse_risk",
]
label_counter: Counter[str] = Counter()
for raw in benchmark_df["label_issue_keys"]:
    label_counter.update(json.loads(raw))
issue_label_report = {
    "phase_id": PHASE_ID,
    "schema_version": SCHEMA_VERSION,
    "label_version": LABEL_VERSION,
    "generated_at": utc_now(),
    "taxonomy": issue_taxonomy,
    "required_issue_keys": required_issue_keys,
    "missing_required_keys": sorted(set(required_issue_keys) - set(issue_taxonomy)),
    "label_positive_counts": {key: int(label_counter.get(key, 0)) for key in sorted(issue_taxonomy)},
    "label_source": "synthetic locked benchmark fixtures; production labels require reviewer replacement before release claims beyond fixture behavior",
}
if issue_label_report["missing_required_keys"]:
    raise AssertionError(f"Missing ATS issue keys: {issue_label_report['missing_required_keys']}")
write_json(ISSUE_LABELS_PATH, issue_label_report)
issue_label_report


{'phase_id': 'phase_19_ats_friendliness_benchmark_scorer',
 'schema_version': 'ats-benchmark-scorer-v1',
 'label_version': 'ats-issue-labels-v1',
 'generated_at': '2026-06-02T04:52:49.361286+00:00',
 'taxonomy': {'parseability_issue': {'description': 'Parser cannot reliably extract meaningful text.',
   'evidence': ['parser_error', 'low extraction coverage', 'empty text']},
  'section_completeness_issue': {'description': 'Core CV sections are missing or not detected.',
   'evidence': ['section_count < 3',
    'missing summary/experience/skills/education']},
  'contact_detection_issue': {'description': 'Contact fields are absent or unverified.',
   'evidence': ['email/phone signal missing when raw text is available']},
  'date_detection_issue': {'description': 'Experience or education dates are absent or inconsistent.',
   'evidence': ['date pattern missing', 'timeline ambiguity']},
  'metric_evidence_issue': {'description': 'Experience lacks quantified impact evidence.',
   'evidence':

## Step 19.4 — Transparent ATS scorer and baseline comparison

### Purpose
Implement a transparent ATS scorer/classifier and compare it against a rule-based baseline.

### Required input
Parser signals, predicted issue keys, benchmark labels, and release thresholds.

### Action
Score each fixture by applying explicit issue penalties and safe fallback rules. Compare the transparent scorer against a naive empty-text baseline.

### Expected output
`reports/phase_19_ats_scorer_metrics.json` with issue precision, recall, score-bucket agreement, and baseline comparison.

### Verification
`atsFriendliness.score` is `0-100`, `detectedIssues` are evidence-backed taxonomy keys, and unsupported/scanned/empty parses receive low-score fallback outputs.


In [4]:
def score_bucket(score: float) -> str:
    if score >= 75:
        return "high"
    if score >= 45:
        return "medium"
    return "low"

issue_penalties = {
    "parseability_issue": 35,
    "empty_parse_risk": 25,
    "unsupported_file_type": 35,
    "section_completeness_issue": 18,
    "metric_evidence_issue": 10,
    "formatting_risk_issue": 18,
    "contact_detection_issue": 8,
    "date_detection_issue": 8,
}


def ats_score(issues: list[str]) -> int:
    if {"parseability_issue", "empty_parse_risk", "unsupported_file_type"} & set(issues):
        return max(0, 35 - sum(issue_penalties.get(issue, 0) for issue in issues if issue not in {"parseability_issue", "empty_parse_risk"}) // 3)
    return int(max(0, min(100, 100 - sum(issue_penalties.get(issue, 0) for issue in issues))))

benchmark_df["predicted_issue_keys"] = benchmark_df["predicted_parser_issues"]
benchmark_df["predicted_ats_score"] = benchmark_df["predicted_issue_keys"].map(ats_score)
benchmark_df["predicted_ats_bucket"] = benchmark_df["predicted_ats_score"].map(score_bucket)
benchmark_df["baseline_issue_keys"] = benchmark_df.apply(lambda row: ["parseability_issue", "empty_parse_risk"] if row["empty_text"] else [], axis=1)
benchmark_df["baseline_ats_score"] = benchmark_df["baseline_issue_keys"].map(ats_score)
benchmark_df["baseline_ats_bucket"] = benchmark_df["baseline_ats_score"].map(score_bucket)

all_issue_keys = sorted(set(issue_taxonomy) | set().union(*(set(xs) for xs in benchmark_df["predicted_issue_keys"])))

def issue_metrics(pred_col: str) -> dict[str, Any]:
    per_issue = {}
    precisions = []
    recalls = []
    for issue in all_issue_keys:
        pred = benchmark_df[pred_col].map(lambda xs: issue in xs)
        label = benchmark_df["label_issue_keys"].map(lambda s: issue in json.loads(s))
        tp = int((pred & label).sum())
        fp = int((pred & ~label).sum())
        fn = int((~pred & label).sum())
        precision = tp / (tp + fp) if (tp + fp) else None
        recall = tp / (tp + fn) if (tp + fn) else None
        if precision is not None:
            precisions.append(precision)
        if recall is not None:
            recalls.append(recall)
        per_issue[issue] = {"tp": tp, "fp": fp, "fn": fn, "precision": precision, "recall": recall}
    return {
        "per_issue": per_issue,
        "macro_issue_precision": float(np.mean(precisions)) if precisions else None,
        "macro_issue_recall": float(np.mean(recalls)) if recalls else None,
    }

transparent_issue_metrics = issue_metrics("predicted_issue_keys")
baseline_issue_metrics = issue_metrics("baseline_issue_keys")
transparent_bucket_agreement = float((benchmark_df["predicted_ats_bucket"] == benchmark_df["label_ats_bucket"]).mean())
baseline_bucket_agreement = float((benchmark_df["baseline_ats_bucket"] == benchmark_df["label_ats_bucket"]).mean())
critical_recall = parser_metrics["overall"]["critical_parse_failure_recall"]
scorer_metrics = {
    "phase_id": PHASE_ID,
    "schema_version": SCHEMA_VERSION,
    "generated_at": utc_now(),
    "transparent_scorer": {
        **transparent_issue_metrics,
        "bucket_agreement": transparent_bucket_agreement,
        "score_min": int(benchmark_df["predicted_ats_score"].min()),
        "score_max": int(benchmark_df["predicted_ats_score"].max()),
    },
    "baseline_empty_text_only": {
        **baseline_issue_metrics,
        "bucket_agreement": baseline_bucket_agreement,
    },
    "release_thresholds": RELEASE_THRESHOLDS,
    "release_gate": {
        "macro_issue_precision_passed": transparent_issue_metrics["macro_issue_precision"] >= RELEASE_THRESHOLDS["macro_issue_precision"],
        "macro_issue_recall_passed": transparent_issue_metrics["macro_issue_recall"] >= RELEASE_THRESHOLDS["macro_issue_recall"],
        "bucket_agreement_passed": transparent_bucket_agreement >= RELEASE_THRESHOLDS["bucket_agreement"],
        "critical_parse_failure_recall_passed": critical_recall >= RELEASE_THRESHOLDS["critical_parse_failure_recall"],
        "beats_baseline_bucket_agreement": transparent_bucket_agreement > baseline_bucket_agreement,
    },
}
write_json(SCORER_METRICS_PATH, scorer_metrics)
scorer_metrics


{'phase_id': 'phase_19_ats_friendliness_benchmark_scorer',
 'schema_version': 'ats-benchmark-scorer-v1',
 'generated_at': '2026-06-02T04:52:49.394242+00:00',
 'transparent_scorer': {'per_issue': {'contact_detection_issue': {'tp': 0,
    'fp': 0,
    'fn': 0,
    'precision': None,
    'recall': None},
   'date_detection_issue': {'tp': 0,
    'fp': 0,
    'fn': 0,
    'precision': None,
    'recall': None},
   'empty_parse_risk': {'tp': 3,
    'fp': 0,
    'fn': 0,
    'precision': 1.0,
    'recall': 1.0},
   'formatting_risk_issue': {'tp': 9,
    'fp': 0,
    'fn': 0,
    'precision': 1.0,
    'recall': 1.0},
   'metric_evidence_issue': {'tp': 6,
    'fp': 0,
    'fn': 0,
    'precision': 1.0,
    'recall': 1.0},
   'parseability_issue': {'tp': 3,
    'fp': 0,
    'fn': 0,
    'precision': 1.0,
    'recall': 1.0},
   'section_completeness_issue': {'tp': 3,
    'fp': 0,
    'fn': 0,
    'precision': 1.0,
    'recall': 1.0},
   'unsupported_file_type': {'tp': 0,
    'fp': 0,
    'fn': 0,

## Step 19.5 — Issue errors, fallback behavior, and phase gate

### Purpose
Report precision, recall, bucket agreement, issue-level errors, and fallback behavior.

### Required input
Scorer predictions, labels, parser metrics, and release thresholds.

### Action
Export false positives, false negatives, fallback examples, output contract examples, and the final phase report.

### Expected output
`reports/phase_19_issue_errors_fallbacks.json` and `reports/phase_19_ats_friendliness_benchmark_scorer.json`.

### Verification
Acceptance criteria are evaluated explicitly; fallback outputs use evidence-backed detected issues and safe low scores for critical parse failures.


In [5]:
def output_contract(row: pd.Series) -> dict[str, Any]:
    issues = list(row["predicted_issue_keys"])
    return {
        "fixtureId": row["fixture_id"],
        "atsFriendliness": {
            "score": int(row["predicted_ats_score"]),
            "detectedIssues": issues,
            "evidence": {
                "extractedCharacterCount": int(row["extracted_character_count"]),
                "extractionCoverage": round(float(row["extraction_coverage"]), 4),
                "sectionCount": int(row["section_count"]),
                "layoutRisk": row["layout_risk"],
                "parserError": bool(row["parser_error"]),
            },
            "fallback": bool({"parseability_issue", "empty_parse_risk", "unsupported_file_type"} & set(issues)),
        },
        "model": {"name": PHASE_ID, "version": SCHEMA_VERSION},
    }

error_records = []
for issue in all_issue_keys:
    for _, row in benchmark_df.iterrows():
        pred = issue in row["predicted_issue_keys"]
        label = issue in json.loads(row["label_issue_keys"])
        if pred != label:
            error_records.append({
                "fixture_id": row["fixture_id"],
                "case_family": row["case_family"],
                "issue": issue,
                "error_type": "false_positive" if pred and not label else "false_negative",
                "predicted_issues": row["predicted_issue_keys"],
                "label_issues": json.loads(row["label_issue_keys"]),
            })

fallback_examples = [output_contract(row) for _, row in benchmark_df[benchmark_df["critical_parse_failure_pred"]].iterrows()]
contract_examples = [output_contract(row) for _, row in benchmark_df.groupby("case_family", sort=True).head(1).iterrows()]
errors_fallbacks = {
    "phase_id": PHASE_ID,
    "schema_version": SCHEMA_VERSION,
    "generated_at": utc_now(),
    "issue_error_count": len(error_records),
    "issue_errors": error_records,
    "fallback_examples": fallback_examples,
    "output_contract_examples": contract_examples,
    "forbidden_output_fields": ["topActionables", "sectionReviews", "overallImpression", "jobFitAlignment"],
}
write_json(ERRORS_FALLBACKS_PATH, errors_fallbacks)

empty_text_rate_supported = parser_metrics["overall"]["empty_text_rate_supported"]
release_gate = scorer_metrics["release_gate"]
acceptance_criteria = {
    "ats_precision_recall_and_bucket_agreement_meet_release_thresholds": bool(
        release_gate["macro_issue_precision_passed"]
        and release_gate["macro_issue_recall_passed"]
        and release_gate["bucket_agreement_passed"]
    ),
    "empty_text_rate_on_supported_documents_under_release_threshold": bool(empty_text_rate_supported <= RELEASE_THRESHOLDS["empty_text_rate_supported_max"]),
    "unsupported_scanned_empty_parse_cases_trigger_safe_fallback_outputs": bool(
        release_gate["critical_parse_failure_recall_passed"] and len(fallback_examples) >= 3
    ),
    "ats_friendliness_score_and_detected_issues_are_evidence_backed": bool(
        all(set(row["predicted_issue_keys"]).issubset(issue_taxonomy) for _, row in benchmark_df.iterrows())
        and all(0 <= int(score) <= 100 for score in benchmark_df["predicted_ats_score"])
    ),
}
phase_status = "complete" if all(acceptance_criteria.values()) else "blocked"
phase_report = {
    "phase_id": PHASE_ID,
    "schema_version": SCHEMA_VERSION,
    "status": phase_status,
    "generated_at": utc_now(),
    "acceptance_criteria": acceptance_criteria,
    "benchmark_manifest": {"path": rel(BENCHMARK_MANIFEST_PATH), "sha256": sha256_file(BENCHMARK_MANIFEST_PATH)},
    "parser_metrics": {"path": rel(PARSER_METRICS_PATH), "sha256": sha256_file(PARSER_METRICS_PATH)},
    "issue_labels": {"path": rel(ISSUE_LABELS_PATH), "sha256": sha256_file(ISSUE_LABELS_PATH)},
    "scorer_metrics": {"path": rel(SCORER_METRICS_PATH), "sha256": sha256_file(SCORER_METRICS_PATH)},
    "errors_fallbacks": {"path": rel(ERRORS_FALLBACKS_PATH), "sha256": sha256_file(ERRORS_FALLBACKS_PATH)},
    "artifact": {"path": rel(BENCHMARK_CSV_PATH), "sha256": sha256_file(BENCHMARK_CSV_PATH), "row_count": int(len(benchmark_df))},
    "selected_scorer": "transparent_rule_based_ats_scorer",
    "release_thresholds": RELEASE_THRESHOLDS,
    "notes": [
        "Synthetic fixtures contain no personal CV data.",
        "Backend/API wrapper remains owner of product copy and final response mapping.",
        "Real CV benchmark documents can replace synthetic fixtures by preserving the manifest schema.",
    ],
    "blockers": [] if phase_status == "complete" else ["One or more release gates failed; inspect scorer metrics."],
}
write_json(PHASE_REPORT_PATH, phase_report)
phase_report


{'phase_id': 'phase_19_ats_friendliness_benchmark_scorer',
 'schema_version': 'ats-benchmark-scorer-v1',
 'status': 'complete',
 'generated_at': '2026-06-02T04:52:49.423110+00:00',
 'acceptance_criteria': {'ats_precision_recall_and_bucket_agreement_meet_release_thresholds': True,
  'empty_text_rate_on_supported_documents_under_release_threshold': True,
  'unsupported_scanned_empty_parse_cases_trigger_safe_fallback_outputs': True,
  'ats_friendliness_score_and_detected_issues_are_evidence_backed': True},
 'benchmark_manifest': {'path': 'reports/phase_19_cv_benchmark_manifest.json',
  'sha256': 'b09f2fd20d387586b5143cb1ff3791c1b56faf4bfcf0a8c5b94134a656ff473d'},
 'parser_metrics': {'path': 'reports/phase_19_parser_benchmark_metrics.json',
  'sha256': 'c6138d7e1302659f284081564ff3d6ccdc24ba453c046cb79d15d5982219891e'},
 'issue_labels': {'path': 'reports/phase_19_ats_issue_labels.json',
  'sha256': 'fd9192944caef47bde6424e5e6aef3b45325db4b39aa532d7504ff0cb908747e'},
 'scorer_metrics': {'pa